# Practice Assessment 10: The Unhappy Sampler

**Topic**: Bayesian Workflow & MCMC Diagnostics
**Chapter**: 10
**Goal**: Diagnose sampling problems (divergences) and fix them using model re-parameterization.

---

## 1. Scenario: The Soil Quality Mystery

You are an agricultural data scientist modeling the relationship between soil heterogeneity (`v`) and crop yield (`x`) across different experimental plots.

You have designed a hierarchical model that you believe perfectly captures the physics of the soil. However, when you try to fit it using PyMC, the sampler complains. It runs slowly, the effective sample size is low, and worst of all, you see ominous warnings about "Divergences".

Your supervisor says: *> "Just run it for more iterations!"*

You suspect that simply running it longer won't help. You need to investigate the geometry of the posterior.

## 2. The Data (The Devil's Funnel)

In this exercise, we won't load external data. Instead, we will try to sample from a known mathematical distribution that is notorious for breaking MCMC samplers: **Neal's Funnel** (often called the Devil's Funnel).

The model is defined as:
$$ v \sim \text{Normal}(0, 3) $$
$$ x \sim \text{Normal}(0, e^{v/2}) $$

(Note: Standard deviation is $e^{v/2}$, so variance is $e^v$).

This simple 2-parameter distribution mimics the structure of many hierarchical models where a group-level variance parameter (`v`) controls the spread of individual inputs (`x`).

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

print(f"PyMC version: {pm.__version__}")

---

## Task 1: The Bad Model (Centered Parameterization)

Implement the model exactly as written above. This is called the **Centered Parameterization** because `x` is defined conditionally on `v`.

1. Define the model in PyMC.
2. Sample using `pm.sample(draws=1000, tune=1000, chains=2, random_seed=42)`.
3. **Important**: Pay attention to the output messages. Do you see any warnings?

In [ ]:
with pm.Model() as model_centered:
    # v ~ Normal(0, 3)
    v = pm.Normal("v", mu=0, sigma=3)
    
    # x ~ Normal(0, exp(v/2))
    # Hint: Use pm.math.exp(v/2) for the sigma
    x = pm.Normal("x", mu=0, sigma=pm.math.exp(v/2))
    
    # Sample
    trace_centered = pm.sample(draws=1000, tune=1000, chains=2, random_seed=42)

### Question 1
Did you get divergence warnings? What does a "divergence" mean geometrically? (Think about the skier analogy from the lecture).

*(Write your answer here)*

---

## Task 2: Visual Diagnostics

Use ArviZ to diagnose the health of the chains.

1. Print the summary (`az.summary`) to check `r_hat` and `ess_bulk`.
2. Plot the **Trace Plot** (`az.plot_trace`). Look for "stuck" chains.
3. Plot the **Rank Plot** (`az.plot_rank`). What should it look like if mixing is good?

In [ ]:
# Summary stats
print(az.summary(trace_centered, var_names=["v", "x"]))

# Trace plot
az.plot_trace(trace_centered, var_names=["v", "x"])
plt.show()

# Rank plot
az.plot_rank(trace_centered, var_names=["v", "x"])
plt.show()

### Question 2
Look at the trace plot for `v`. Does it look like a "fuzzy caterpillar"? Describe what you see in the worst performing chain.

*(Write your answer here)*

---

## Task 3: The Fix (Non-Centered Parameterization)

The problem is that the curvature of the funnel is too extreme for the sampler when `v` is small (small variance -> tiny neck of funnel).

We can fix this by **breaking the dependence** in the sampling step. We sample a raw variable from a standard normal, and then transform it deterministically.

**Mathematical transformation**:
$$ v \sim \text{Normal}(0, 3) $$
$$ z \sim \text{Normal}(0, 1) $$
$$ x = z \cdot e^{v/2} $$

Implement this `model_non_centered` in PyMC. Note that `x` is now a `Deterministic` variable, not a stochastic one sampled directly.

In [ ]:
with pm.Model() as model_non_centered:
    # 1. v ~ Normal(0, 3)
    v = pm.Normal("v", mu=0, sigma=3)
    
    # 2. z ~ Normal(0, 1)  <-- This is the helper variable
    z = pm.Normal("z_raw", mu=0, sigma=1)
    
    # 3. Deterministic transformation for x
    # x = z * exp(v/2)
    x = pm.Deterministic("x", z * pm.math.exp(v/2))
    
    # Sample
    trace_non_centered = pm.sample(draws=1000, tune=1000, chains=2, random_seed=42)

---

## Task 4: Comparison

Did it work? Let's check the diagnostics for the new model.

1. Print the summary (`ess_bulk`, `r_hat`).
2. Check for divergences.
3. Visualize the geometry: Use `az.plot_pair` to plot `v` vs `x` for both models.

In [ ]:
print("--- Non-Centered Model Summary ---")
print(az.summary(trace_non_centered, var_names=["v", "x"]))

div_count = trace_non_centered.sample_stats.diverging.sum().values
print(f"\nTotal Divergences: {div_count}")

# Geometric Comparison
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True, sharex=True)

az.plot_pair(trace_centered, var_names=["v", "x"], kind="scatter", ax=ax[0], 
             divergences=True, scatter_kwargs={'alpha': 0.1})
ax[0].set_title("Centered (With Divergences)")

az.plot_pair(trace_non_centered, var_names=["v", "x"], kind="scatter", ax=ax[1],
             divergences=True, scatter_kwargs={'alpha': 0.1})
ax[1].set_title("Non-Centered (Healthy)")

plt.show()

### Question 3
Compare the two scatter plots above. 
- Where does the Centered model fail to explore? (Look at the neck of the funnel where `v` is small).
- Why does the Non-Centered model succeed in exploring that region?

*(Write your answer here)*